# Tahap 2 - Case Representation

Project: Case-Based Reasoning untuk Pidana Umum - Pencurian di PN Tangerang

Notebook ini digunakan sebagai bagian dari pipeline CBR.


In [1]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("..").resolve()

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

inventory_path = PROCESSED_DIR / "case_inventory.csv"

df_inventory = pd.read_csv(inventory_path, dtype=str).fillna("")

print("Jumlah data inventory:", len(df_inventory))
print("Jumlah file txt di data/raw:", len(list(RAW_DIR.glob("*.txt"))))

df_inventory[[
    "case_id",
    "no_perkara",
    "tanggal_putusan",
    "pengadilan",
    "jenis_perkara",
    "raw_file",
    "status_download",
    "jumlah_kata"
]].head(40)

Jumlah data inventory: 40
Jumlah file txt di data/raw: 40


,case_id,no_perkara,tanggal_putusan,pengadilan,jenis_perkara,raw_file,status_download,jumlah_kata
0,case_001,1022/Pid.B/2010/PN.TNG,14 Juli 2010,PN Tangerang,Pidana Umum - Pencurian,case_001.txt,berhasil_list_html,80
1,case_002,497 / PID.B / 2014 / PN.TNG.,19 Mei 2014,PN Tangerang,Pidana Umum - Pencurian,case_002.txt,berhasil_list_html,322
2,case_003,1885 /Pid.B/2011/PN.TNG,15 Desember 2011,PN Tangerang,Pidana Umum - Pencurian,case_003.txt,berhasil_list_html,123
3,case_004,1073/Pid.B/2019/PN Tng,10 Juli 2019,PN Tangerang,Pidana Umum - Pencurian,case_004.txt,berhasil_list_html,213
4,case_005,678/ PID.B/ 2011/ PN TNG,8 Mei 2012,PN Tangerang,Pidana Umum - Pencurian,case_005.txt,berhasil_list_html,342
5,case_006,1527/Pid.B/2014/PN.TNG,16 September 2014,PN Tangerang,Pidana Umum - Pencurian,case_006.txt,berhasil_list_html,276
6,case_007,834 / Pid.B / 2017 / PN.TNG.,15 Juni 2017,PN Tangerang,Pidana Umum - Pencurian,case_007.txt,berhasil_list_html,263
7,case_008,2000/Pid.B/2017/PN.Tng,16 Nopember 2017,PN Tangerang,Pidana Umum - Pencurian,case_008.txt,berhasil_list_html,349
8,case_009,2497/Pid.B/2018/PN Tng,10 Januari 2019,PN Tangerang,Pidana Umum - Pencurian,case_009.txt,berhasil_list_html,73
9,case_010,329/Pid.B/2019/PN Tng,1 April 2019,PN Tangerang,Pidana Umum - Pencurian,case_010.txt,berhasil_list_html,335


In [4]:
from pathlib import Path
import pandas as pd
import re
import shutil

BASE_DIR = Path("..").resolve()

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

inventory_path = PROCESSED_DIR / "case_inventory.csv"
cases_path = PROCESSED_DIR / "cases.csv"
backup_cases_path = PROCESSED_DIR / "cases_before_revision.csv"

if cases_path.exists():
    shutil.copy2(cases_path, backup_cases_path)
    print("Backup cases lama dibuat:", backup_cases_path)

df_inventory = pd.read_csv(inventory_path, dtype=str).fillna("")


def normalize_text(text):
    text = str(text)
    text = text.replace("\r", " ")
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def extract_actual_category(text):
    text = normalize_text(text)
    
    match = re.search(
        r"Pengadilan\s+PN\s+TANGERANG\s+(.*?)\s+Putus\s*:",
        text,
        re.IGNORECASE
    )
    
    if match:
        return normalize_text(match.group(1))
    
    return ""


def extract_penuntut_umum(text):
    text = normalize_text(text)
    
    patterns = [
        r"Penuntut\s+Umum\s*:?\s*(.*?)(?=Terdakwa|Putusan PN|Nomor|$)",
        r"Penuntut\s+Umum\s*(.*?)(?=Terdakwa|Putusan PN|Nomor|$)"
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
        if match:
            return normalize_text(match.group(1))[:250]
    
    return ""


def extract_terdakwa(text):
    text = normalize_text(text)
    
    bulan = "Januari|Februari|Maret|April|Mei|Juni|Juli|Agustus|September|Oktober|November|Nopember|Desember"
    
    patterns = [
        rf"Tanggal\s+[0-9]{{1,2}}\s+(?:{bulan})\s+[0-9]{{4}}\s+—\s*(.*?)\s+\d+\s+—\s+\d+",
        r"Terdakwa\s*:?\s*(.*?)(?=Nomor|Tingkat Proses|Klasifikasi|Kata Kunci|Tahun|Hakim|Amar|$)",
        r"—\s*(.*?)\s+\d+\s+—\s+\d+"
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
        if match:
            hasil = normalize_text(match.group(1))
            hasil = hasil.replace("Penuntut Umum:", "")
            return hasil[:300]
    
    return ""


def extract_solution_text(text):
    text = normalize_text(text)
    
    # Pola umum dari card hasil pencarian MA:
    # "... NAMA TERDAKWA 23 — 1 Menyatakan Terdakwa ..."
    match = re.search(r"\b\d+\s+—\s+\d+\s+(.*)$", text)
    
    if match:
        hasil = normalize_text(match.group(1))
        if len(hasil.split()) >= 5:
            return hasil
    
    # Fallback jika tidak menemukan pola angka statistik
    keywords = [
        "M E N G A D I L I",
        "MENGADILI",
        "Menyatakan Terdakwa",
        "Menyatakan bahwa Terdakwa",
        "Menyatakan",
        "Menjatuhkan pidana",
        "Menjatuhkan"
    ]
    
    text_lower = text.lower()
    
    for key in keywords:
        pos = text_lower.find(key.lower())
        if pos != -1:
            return normalize_text(text[pos:])
    
    return text


def extract_pasal(text):
    text = normalize_text(text)
    
    patterns = [
        r"Pasal\s+[0-9]+[A-Za-z]?(?:\s+ayat\s+\([0-9]+\))?(?:\s+ke[-\s]?[0-9]+)?(?:\s+KUHP)?",
        r"Pasal\s+[0-9]+[A-Za-z]?\s+KUHP",
        r"Pasal\s+[0-9]+[A-Za-z]?"
    ]
    
    hasil = []
    
    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        for m in matches:
            m = normalize_text(m)
            if m not in hasil:
                hasil.append(m)
    
    return "; ".join(hasil[:10])


def extract_lama_pidana(solution_text):
    text = normalize_text(solution_text)
    
    patterns = [
        (r"HUKUM\s+(\d+)\s+TAHUN\s+(\d+)\s+BULAN", "tahun_bulan"),
        (r"HUKUM\s+(\d+)\s+TAHUN", "tahun"),
        (r"HUKUM\s+(\d+)\s+BULAN", "bulan"),
        (r"selama\s*(\d+)\s*\([^)]*\)\s*tahun\s*,?\s*(\d+)?\s*(?:\([^)]*\))?\s*bulan?", "tahun_bulan_optional"),
        (r"selama\s*(\d+)\s*\([^)]*\)\s*tahun", "tahun"),
        (r"selama\s*(\d+)\s*\([^)]*\)\s*bulan", "bulan"),
        (r"penjara\s+selama\s*(\d+)\s+tahun\s*(\d+)?\s*bulan?", "tahun_bulan_optional"),
        (r"penjara\s+selama\s*(\d+)\s+bulan", "bulan")
    ]
    
    for pattern, kind in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        
        if match:
            lama_text = normalize_text(match.group(0))
            
            if kind == "tahun_bulan":
                tahun = int(match.group(1))
                bulan = int(match.group(2))
                total_bulan = tahun * 12 + bulan
                return lama_text, total_bulan
            
            if kind == "tahun_bulan_optional":
                tahun = int(match.group(1)) if match.group(1) else 0
                bulan = int(match.group(2)) if match.lastindex and match.group(2) else 0
                total_bulan = tahun * 12 + bulan
                return lama_text, total_bulan
            
            if kind == "tahun":
                tahun = int(match.group(1))
                total_bulan = tahun * 12
                return lama_text, total_bulan
            
            if kind == "bulan":
                bulan = int(match.group(1))
                return lama_text, bulan
    
    return "", None


def make_solution_label(solution_text):
    text_lower = normalize_text(solution_text).lower()
    
    lama_text, total_bulan = extract_lama_pidana(solution_text)
    
    if "bebas" in text_lower:
        return "Bebas"
    
    if "lepas" in text_lower:
        return "Lepas"
    
    if total_bulan is not None:
        if total_bulan <= 6:
            return "Pidana Penjara <= 6 Bulan"
        elif total_bulan <= 12:
            return "Pidana Penjara 7-12 Bulan"
        else:
            return "Pidana Penjara > 12 Bulan"
    
    if any(keyword in text_lower for keyword in [
        "penjara",
        "pidana",
        "bersalah",
        "terbukti",
        "menyatakan terdakwa",
        "menjatuhkan"
    ]):
        return "Pidana Penjara Tidak Teridentifikasi"
    
    return "Lain-lain"


cases = []

for _, row in df_inventory.iterrows():
    case_id = row.get("case_id", "")
    raw_file = row.get("raw_file", "")
    raw_path = RAW_DIR / raw_file
    
    if not raw_path.exists():
        print(f"File tidak ditemukan: {raw_file}")
        continue
    
    text_full = raw_path.read_text(encoding="utf-8", errors="ignore")
    text_full = normalize_text(text_full)
    
    jumlah_kata = len(text_full.split())
    
    actual_category = extract_actual_category(text_full)
    terdakwa = extract_terdakwa(text_full)
    penuntut_umum = extract_penuntut_umum(text_full)
    
    solution_text = extract_solution_text(text_full)
    solution_text = normalize_text(solution_text)
    
    pasal = extract_pasal(solution_text + " " + text_full)
    
    lama_pidana, lama_pidana_bulan = extract_lama_pidana(solution_text)
    solution_label = make_solution_label(solution_text)
    
    # Karena data berasal dari HTML/list, bagian amar/catatan amar direpresentasikan dari excerpt putusan
    amar_lainnya = solution_text
    catatan_amar = solution_text
    
    ringkasan_fakta = solution_text[:1500]
    argumen_hukum = solution_text[:1500]
    
    cases.append({
        "case_id": case_id,
        "no_perkara": row.get("no_perkara", ""),
        "tanggal_putusan": row.get("tanggal_putusan", ""),
        "pengadilan": row.get("pengadilan", "PN Tangerang"),
        "jenis_perkara": row.get("jenis_perkara", "Pidana Umum - Pencurian"),
        "kategori_asli_html": actual_category,
        "penuntut_umum": penuntut_umum,
        "terdakwa": terdakwa,
        "pasal": pasal,
        "hakim_ketua": "",
        "hakim_anggota": "",
        "panitera": "",
        "amar": solution_text,
        "amar_lainnya": amar_lainnya,
        "catatan_amar": catatan_amar,
        "tanggal_musyawarah": "",
        "tanggal_dibacakan": row.get("tanggal_putusan", ""),
        "ringkasan_fakta": ringkasan_fakta,
        "argumen_hukum": argumen_hukum,
        "solution_text": solution_text,
        "solution_label": solution_label,
        "lama_pidana": lama_pidana,
        "lama_pidana_bulan": "" if lama_pidana_bulan is None else lama_pidana_bulan,
        "sumber_url": row.get("sumber_url", ""),
        "raw_file": raw_file,
        "jumlah_kata": jumlah_kata,
        "text_full": text_full
    })

cases_df = pd.DataFrame(cases)

cases_df.to_csv(cases_path, index=False)

print("cases.csv berhasil direvisi.")
print("Lokasi:", cases_path)
print("Jumlah kasus:", len(cases_df))

cases_df[[
    "case_id",
    "no_perkara",
    "terdakwa",
    "pasal",
    "solution_label",
    "lama_pidana",
    "jumlah_kata"
]].head(40)

Backup cases lama dibuat: /home/zack/Penalaran-Komputer-subcpmk-3/data/processed/cases_before_revision.csv
cases.csv berhasil direvisi.
Lokasi: /home/zack/Penalaran-Komputer-subcpmk-3/data/processed/cases.csv
Jumlah kasus: 40


,case_id,no_perkara,terdakwa,pasal,solution_label,lama_pidana,jumlah_kata
0,case_001,1022/Pid.B/2010/PN.TNG,JAMALUDIN Bin SANIF,Pasal 363 ayat (2) KUHP; Pasal 363,Pidana Penjara 7-12 Bulan,selama 1 (satu) tahun,80
1,case_002,497 / PID.B / 2014 / PN.TNG.,AMIN Als UBE Bin UDIN dan ALIP KURNIAWAN Als O...,Pasal 365 ayat (2) ke2 KUHP; Pasal 365,Pidana Penjara Tidak Teridentifikasi,,322
2,case_003,1885 /Pid.B/2011/PN.TNG,MAULANA HASANUDIN als. KEDOK bin ROJALI,,Pidana Penjara 7-12 Bulan,selama 8 (delapan) bulan,123
3,case_004,1073/Pid.B/2019/PN Tng,"REZA VAHLEVI, SH Terdakwa: MUHAMMAD RIKI YAKU...",Pasal 362 KUHP; Pasal 362,Pidana Penjara <= 6 Bulan,selama 4 (empat) bulan,213
4,case_005,678/ PID.B/ 2011/ PN TNG,MOHAMAD HERI Als HERI Bin SAMIR,Pasal 362 KUHP; Pasal 362,Pidana Penjara > 12 Bulan,selama 15 (lima betas) tahun,342
5,case_006,1527/Pid.B/2014/PN.TNG,NUR APRIYANI Binti (Alm) NURDIN,Pasal 362 KUHP; Pasal 362,Pidana Penjara <= 6 Bulan,selama 6 (enam) bulan,276
6,case_007,834 / Pid.B / 2017 / PN.TNG.,DEDY KUMALA BIN SAHLAN,Pasal 362 KUHP; Pasal 362,Pidana Penjara 7-12 Bulan,selama 1 (satu) tahun,263
7,case_008,2000/Pid.B/2017/PN.Tng,YUDI HARYADI Bin BAROZI.,Pasal 362 KUHP; Pasal 362K; Pasal 362,Pidana Penjara 7-12 Bulan,selama 1 (satu) tahun,349
8,case_009,2497/Pid.B/2018/PN Tng,"ROSI PAREME DEWI INDAH, SH Terdakwa: 1.AFRIYA...",,Pidana Penjara Tidak Teridentifikasi,,73
9,case_010,329/Pid.B/2019/PN Tng,TRIADE Terdakwa: 1.ICON SIREGAR Alias ICON Bi...,,Pidana Penjara 7-12 Bulan,selama 1(satu) tahun,335


In [5]:
cases_df = pd.read_csv(cases_path, dtype=str).fillna("")

print("Jumlah kasus:", len(cases_df))
print("Duplikat case_id:", cases_df["case_id"].duplicated().sum())
print("Duplikat no_perkara:", cases_df["no_perkara"].duplicated().sum())

print("\nKolom kosong penting:")
important_cols = [
    "terdakwa",
    "pasal",
    "amar_lainnya",
    "catatan_amar",
    "solution_text",
    "solution_label",
    "lama_pidana"
]

for col in important_cols:
    print(col, ":", (cases_df[col] == "").sum(), "kosong")

print("\nDistribusi solution_label:")
print(cases_df["solution_label"].value_counts())

print("\nStatistik jumlah kata:")
print(cases_df["jumlah_kata"].astype(int).describe())

cases_df[[
    "case_id",
    "no_perkara",
    "terdakwa",
    "pasal",
    "solution_label",
    "lama_pidana",
    "jumlah_kata"
]].head(40)

Jumlah kasus: 40
Duplikat case_id: 0
Duplikat no_perkara: 0

Kolom kosong penting:
terdakwa : 0 kosong
pasal : 29 kosong
amar_lainnya : 0 kosong
catatan_amar : 0 kosong
solution_text : 0 kosong
solution_label : 0 kosong
lama_pidana : 15 kosong

Distribusi solution_label:
solution_label
Pidana Penjara Tidak Teridentifikasi    15
Pidana Penjara 7-12 Bulan               11
Pidana Penjara > 12 Bulan               11
Pidana Penjara <= 6 Bulan                3
Name: count, dtype: int64

Statistik jumlah kata:
count     40.000000
mean     193.875000
std      116.856736
min       58.000000
25%       97.000000
50%      115.500000
75%      323.750000
max      373.000000
Name: jumlah_kata, dtype: float64


,case_id,no_perkara,terdakwa,pasal,solution_label,lama_pidana,jumlah_kata
0,case_001,1022/Pid.B/2010/PN.TNG,JAMALUDIN Bin SANIF,Pasal 363 ayat (2) KUHP; Pasal 363,Pidana Penjara 7-12 Bulan,selama 1 (satu) tahun,80
1,case_002,497 / PID.B / 2014 / PN.TNG.,AMIN Als UBE Bin UDIN dan ALIP KURNIAWAN Als O...,Pasal 365 ayat (2) ke2 KUHP; Pasal 365,Pidana Penjara Tidak Teridentifikasi,,322
2,case_003,1885 /Pid.B/2011/PN.TNG,MAULANA HASANUDIN als. KEDOK bin ROJALI,,Pidana Penjara 7-12 Bulan,selama 8 (delapan) bulan,123
3,case_004,1073/Pid.B/2019/PN Tng,"REZA VAHLEVI, SH Terdakwa: MUHAMMAD RIKI YAKU...",Pasal 362 KUHP; Pasal 362,Pidana Penjara <= 6 Bulan,selama 4 (empat) bulan,213
4,case_005,678/ PID.B/ 2011/ PN TNG,MOHAMAD HERI Als HERI Bin SAMIR,Pasal 362 KUHP; Pasal 362,Pidana Penjara > 12 Bulan,selama 15 (lima betas) tahun,342
5,case_006,1527/Pid.B/2014/PN.TNG,NUR APRIYANI Binti (Alm) NURDIN,Pasal 362 KUHP; Pasal 362,Pidana Penjara <= 6 Bulan,selama 6 (enam) bulan,276
6,case_007,834 / Pid.B / 2017 / PN.TNG.,DEDY KUMALA BIN SAHLAN,Pasal 362 KUHP; Pasal 362,Pidana Penjara 7-12 Bulan,selama 1 (satu) tahun,263
7,case_008,2000/Pid.B/2017/PN.Tng,YUDI HARYADI Bin BAROZI.,Pasal 362 KUHP; Pasal 362K; Pasal 362,Pidana Penjara 7-12 Bulan,selama 1 (satu) tahun,349
8,case_009,2497/Pid.B/2018/PN Tng,"ROSI PAREME DEWI INDAH, SH Terdakwa: 1.AFRIYA...",,Pidana Penjara Tidak Teridentifikasi,,73
9,case_010,329/Pid.B/2019/PN Tng,TRIADE Terdakwa: 1.ICON SIREGAR Alias ICON Bi...,,Pidana Penjara 7-12 Bulan,selama 1(satu) tahun,335
